# Single CSV Evaluation Notebook

这个 notebook 只评估一个 CSV 文件。输入 CSV 需要有 `seq` 列；如果有 `label` 列，就会计算 metrics。输出 CSV 会保留原始列，并增加：

```text
predict_label
predict_prob
```


## Step 0. 定位项目根目录


In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "lamar_m6a").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "lamar_m6a").exists():
    raise RuntimeError("Cannot find project root containing lamar_m6a/. Run this notebook inside the project directory.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


PosixPath('/picb/rnasys2/lidaoqian/lamar_m6a')

## Step 1. 参数设置


In [2]:
from pathlib import Path

MODE = "full"
CELL = "cell5"
LENGTH = 801
SPLIT = "test"

INPUT_CSV = PROJECT_ROOT / "data" / "processed" / "full_splits" / f"{SPLIT}_m6a_full_{CELL}_{LENGTH}.csv"
OUTPUT_CSV = PROJECT_ROOT / "outputs" / "predictions" / "single_csv" / MODE / f"{CELL}_{LENGTH}_{SPLIT}_pred.csv"
METRICS_JSON = PROJECT_ROOT / "outputs" / "metrics" / "single_csv" / MODE / f"{CELL}_{LENGTH}_{SPLIT}.json"
METRICS_CSV = PROJECT_ROOT / "outputs" / "metrics" / "single_csv" / MODE / f"{CELL}_{LENGTH}_{SPLIT}.csv"

CHECKPOINT_PATH = Path("/picb/rnasys2/lidaoqian/lamar_m6a/outputs/finetune/full/cell5-801-bs16-acc4/checkpoint-10100")
TOKENIZER_PATH = PROJECT_ROOT / "tokenizer" / "single_nucleotide"
MODEL_CONFIG = PROJECT_ROOT / "configs" / "config_150M.json"
MODEL_CLASS = "scripts.sequence_classification_patch:EsmForSequenceClassification"

MODEL_MAX_LENGTH = LENGTH + 2
BATCH_SIZE = 128
DEVICE = None

INPUT_CSV, CHECKPOINT_PATH


(PosixPath('/picb/rnasys2/lidaoqian/lamar_m6a/data/processed/full_splits/test_m6a_full_cell5_801.csv'),
 PosixPath('/picb/rnasys2/lidaoqian/lamar_m6a/outputs/finetune/full/cell5-801-bs16-acc4/checkpoint-10100'))

## Step 2. 导入函数


In [3]:
from argparse import Namespace
from lamar_m6a.evaluation.evaluate_single_csv import predict_csv


## Step 3. 检查路径


In [4]:
for path in [INPUT_CSV, CHECKPOINT_PATH, TOKENIZER_PATH, MODEL_CONFIG]:
    print(path, path.exists())


/picb/rnasys2/lidaoqian/lamar_m6a/data/processed/full_splits/test_m6a_full_cell5_801.csv True
/picb/rnasys2/lidaoqian/lamar_m6a/outputs/finetune/full/cell5-801-bs16-acc4/checkpoint-10100 True
/picb/rnasys2/lidaoqian/lamar_m6a/tokenizer/single_nucleotide True
/picb/rnasys2/lidaoqian/lamar_m6a/configs/config_150M.json True


## Step 4. 预测并保存结果


In [5]:
args = Namespace(
    input_csv=INPUT_CSV,
    output_csv=OUTPUT_CSV,
    checkpoint_path=CHECKPOINT_PATH,
    metrics_json=METRICS_JSON,
    metrics_csv=METRICS_CSV,
    length=LENGTH,
    tokenizer_path=TOKENIZER_PATH,
    model_config=MODEL_CONFIG,
    model_class=MODEL_CLASS,
    model_max_length=MODEL_MAX_LENGTH,
    padding_side="left",
    use_fast_tokenizer=False,
    num_labels=2,
    token_dropout=False,
    positional_embedding_type="rotary",
    hidden_size=768,
    intermediate_size=3072,
    num_attention_heads=12,
    num_hidden_layers=12,
    head_type="Linear",
    kernel_sizes=None,
    ocs=None,
    strict_load=True,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

metrics = predict_csv(args)
metrics


/opt/anaconda3/envs/torch201/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
test_m6a_full_cell5_801.csv:   2%|▉                                                 | 62/3484 [02:52<2:38:31,  2.78s/it]


KeyboardInterrupt: 